# Step 7 Runner — UPR-CRE v0.2 Confidence Filtering / Relation Curriculum

This notebook **does not edit source code**. Apply the Step 7 source-code snippets to your GitHub branch first, then run this notebook.

It runs two v0.2 filter/curriculum settings on Kaggle T4x2 and compares them with Step 6 references.


## Design check before running

Step 6 showed that v0.1 improved relation quality but did not improve 15-epoch Rank-1/mAP. A safe next test is not to change the backbone or loss, but to filter low-confidence hard pseudo-relations early in phase 2 and gradually relax the filter. If relation quality helps, this should reduce early pseudo-label noise. If it does not help, we should stop tuning v0.1/v0.2 and move to a different mechanism.


## Block 0 — Configuration

Change only this block if needed. The default RegDB source matches your Kaggle dataset mount.

In [1]:
from pathlib import Path
from datetime import datetime
import os, json, subprocess, time, shutil, re, csv

CFG = {
    "GITHUB_OWNER": "TranTruongMMCII",
    "REPO_NAME": "UIT.CS2309",
    "BRANCH": "feature/upr-cre",
    "WORK_DIR": "/kaggle/working",

    "DATA_ROOT": "/kaggle/working/VIREID_Dataset",
    "REGDB_SOURCE": "/kaggle/input/datasets/xqq027/reg-db/RegDB",
    "PHASE1_CKPT_PATH": "/kaggle/input/datasets/truongtranmmcii/uit-cs2309-checkpoint/phase1_model_5.pth",

    "SEED": 1,
    "TRIAL": 1,
    "STAGE2_EPOCH": 15,
    "MILESTONES": "8 12",
    "BATCH_PIDNUM": 5,
    "PID_NUMSAMPLE": 4,
    "TEST_BATCH": 64,
    "NUM_WORKERS": 2,
    "LR": 0.00045,

    # UPR v0.1 base used by both v0.2 runs
    "UPR_BETA": 0.1,
    "UPR_GAMMA": 0.0,
    "UPR_MARGIN_WEIGHT": 1.0,
    "UPR_WARMUP_EPOCH": 2,

    # Step 7 v0.2 configs
    "RUN_A_TAG": "upr_v02_b01_g00_filter075to100_p2s15",
    "RUN_A_FILTER_START_RATIO": 0.75,
    "RUN_A_FILTER_END_RATIO": 1.0,
    "RUN_A_FILTER_START_EPOCH": 2,
    "RUN_A_FILTER_END_EPOCH": 10,
    "RUN_A_FILTER_MIN_PAIRS": 40,

    "RUN_B_TAG": "upr_v02_b01_g00_filter055to100_p2s15",
    "RUN_B_FILTER_START_RATIO": 0.55,
    "RUN_B_FILTER_END_RATIO": 1.0,
    "RUN_B_FILTER_START_EPOCH": 2,
    "RUN_B_FILTER_END_EPOCH": 10,
    "RUN_B_FILTER_MIN_PAIRS": 40,

    # Step 6 references for comparison only
    "REF_BASELINE_P2S15_R1": 0.5816,
    "REF_BASELINE_P2S15_MAP": 0.5467,
    "REF_BASELINE_P2S15_MINP": 0.4269,
    "REF_UPR_V01_P2S15_R1": 0.5607,
    "REF_UPR_V01_P2S15_MAP": 0.5408,
    "REF_UPR_V01_P2S15_MINP": 0.4299,
}

RUN_SUFFIX = datetime.utcnow().strftime("step7_%Y%m%d_%H%M%S")
repo_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"]
wsl_dir = repo_dir / "WSL_ReID"
print("RUN_SUFFIX =", RUN_SUFFIX)
print("repo_dir =", repo_dir)
print("wsl_dir =", wsl_dir)
print("checkpoint exists:", Path(CFG["PHASE1_CKPT_PATH"]).exists())


RUN_SUFFIX = step7_20260623_135458
repo_dir = /kaggle/working/UIT.CS2309
wsl_dir = /kaggle/working/UIT.CS2309/WSL_ReID
checkpoint exists: True


/tmp/ipykernel_58/1209593256.py:55: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_SUFFIX = datetime.utcnow().strftime("step7_%Y%m%d_%H%M%S")


## Block 1 — Pull repo and checkout branch

This block resets the Kaggle copy to `origin/feature/upr-cre`. It does not change code.

In [2]:
repo_url = f"https://github.com/{CFG['GITHUB_OWNER']}/{CFG['REPO_NAME']}.git"
repo_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"]

if not repo_dir.exists():
    subprocess.run(["git", "clone", "--branch", CFG["BRANCH"], repo_url, str(repo_dir)], check=True)
else:
    subprocess.run(["git", "fetch", "origin", CFG["BRANCH"]], cwd=repo_dir, check=True)
    subprocess.run(["git", "checkout", CFG["BRANCH"]], cwd=repo_dir, check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{CFG['BRANCH']}"], cwd=repo_dir, check=True)

commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=repo_dir).decode().strip()
print("Commit:", commit)
subprocess.run(["git", "status", "--short"], cwd=repo_dir, check=True)


Cloning into '/kaggle/working/UIT.CS2309'...


Commit: f63d8f5


CompletedProcess(args=['git', 'status', '--short'], returncode=0)

## Block 2 — Install Kaggle requirements and prepare RegDB

This uses the scripts already committed in your repo. It also checks that two GPUs are visible.

In [3]:
wsl_dir = repo_dir / "WSL_ReID"
subprocess.run("pip install -q -r requirements-kaggle.txt", cwd=wsl_dir, shell=True, check=True)
subprocess.run(["python", "scripts/apply_kaggle_compat_patches.py"], cwd=wsl_dir, check=True)

prepare_cmd = ["python", "scripts/prepare_regdb_kaggle.py", "--data-root", CFG["DATA_ROOT"]]
if CFG["REGDB_SOURCE"]:
    prepare_cmd += ["--regdb-source", CFG["REGDB_SOURCE"]]
subprocess.run(prepare_cmd, cwd=wsl_dir, check=True)
subprocess.run(["python", "scripts/check_kaggle_env.py", "--data-root", CFG["DATA_ROOT"]], cwd=wsl_dir, check=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 722.4 kB/s eta 0:00:00
No compatibility patches were needed.
RegDB source: /kaggle/input/datasets/xqq027/reg-db/RegDB
RegDB prepared at: /kaggle/working/VIREID_Dataset/RegDB
Use training argument: --data-path /kaggle/working/VIREID_Dataset
=== PyTorch / GPU ===
torch: 2.10.0+cu128
cuda available: True
gpu count: 2
gpu 0: Tesla T4
gpu 1: Tesla T4

=== RegDB layout ===
/kaggle/working/VIREID_Dataset/RegDB/idx/train_visible_1.txt OK
/kaggle/working/VIREID_Dataset/RegDB/idx/train_thermal_1.txt OK
/kaggle/working/VIREID_Dataset/RegDB/idx/test_visible_1.txt OK
/kaggle/working/VIREID_Dataset/RegDB/idx/test_thermal_1.txt OK

 train_visible_1.txt
first line: Visible/285/male_back_v_05528_285.bmp 0
image: /kaggle/working/VIREID_Dataset/RegDB/Visible/285/male_back_v_05528_285.bmp
exists: True
image size: (64, 128) mode: RGB label: 0

 train_thermal_1.txt
first line: Thermal/285/male_back_t_05528_285.bmp 0
image: /kaggle/working/VIREID_Datas

CompletedProcess(args=['python', 'scripts/check_kaggle_env.py', '--data-root', '/kaggle/working/VIREID_Dataset'], returncode=0)

## Block 3 — Static checks

This checks the code/scripts needed for Step 5. It does not check markdown docs.

In [4]:
required = [
    "main.py", "wsl.py", "task/train.py", "relation_metrics.py",
    "scripts/collect_relation_stats.py", "scripts/run_regdb_upr_v02_filter.sh",
]
missing = [p for p in required if not (wsl_dir / p).exists()]
if missing:
    raise FileNotFoundError("Missing required Step 7 files: " + ", ".join(missing))

markers = {
    "main.py": ["--upr-filter", "--upr-filter-start-epoch", "--upr-filter-end-epoch", "--upr-filter-start-ratio", "--upr-filter-end-ratio", "--upr-filter-min-pairs"],
    "wsl.py": ["_upr_filter_keep_ratio", "_pair_confidence_scores", "_apply_upr_pair_filter", "upr_filter_start_ratio", "score_matrix = np.asarray", "last_upr_filter_stats"],
    "scripts/run_regdb_upr_v02_filter.sh": ["--upr-filter", "--upr-filter-start-ratio", "--upr-filter-end-ratio", "--upr-beta", "--upr-gamma", "--csv-output"],
}
for rel, ms in markers.items():
    text = (wsl_dir / rel).read_text()
    for m in ms:
        if m not in text:
            raise RuntimeError(f"Missing marker {m!r} in {rel}")

subprocess.run(["python", "-m", "py_compile", "main.py", "wsl.py", "relation_metrics.py", "scripts/collect_relation_stats.py"], cwd=wsl_dir, check=True)
subprocess.run(["bash", "-n", "scripts/run_regdb_upr_v02_filter.sh"], cwd=wsl_dir, check=True)
print("Step 7 static checks OK.")


Step 7 static checks OK.


## Block 4 — Locate or extract phase1 checkpoint

This is the only non-parallel part. It trains phase 1 once, saves `model_5.pth`, and uses that checkpoint for both phase-2 jobs.

Expected time on T4: roughly similar to your previous phase-1 run.

In [5]:
phase1_ckpt = Path(CFG["PHASE1_CKPT_PATH"])
if not phase1_ckpt.exists():
    candidates = sorted(Path("/kaggle/input").rglob("phase1_model_5.pth")) + sorted(Path("/kaggle/working").rglob("phase1_model_5.pth"))
    if not candidates:
        raise FileNotFoundError("No phase1_model_5.pth found. Upload checkpoint as Kaggle Input or update CFG['PHASE1_CKPT_PATH'].")
    phase1_ckpt = candidates[0]
print("Using checkpoint:", phase1_ckpt)
print("Size MB:", phase1_ckpt.stat().st_size / (1024*1024))


Using checkpoint: /kaggle/input/datasets/truongtranmmcii/uit-cs2309-checkpoint/phase1_model_5.pth
Size MB: 94.88339710235596


## Block 5 — Start two Step 7 runs on T4x2.

Run this after Block 4. It finds the `model_5.pth` created by the fresh phase-1 run.

In [6]:
run_logs_dir = Path("/kaggle/working/run_logs")
run_logs_dir.mkdir(parents=True, exist_ok=True)
commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=repo_dir).decode().strip()

configs = [
    {"tag": CFG["RUN_A_TAG"], "gpu": 0, "start_ratio": CFG["RUN_A_FILTER_START_RATIO"], "end_ratio": CFG["RUN_A_FILTER_END_RATIO"], "start_epoch": CFG["RUN_A_FILTER_START_EPOCH"], "end_epoch": CFG["RUN_A_FILTER_END_EPOCH"], "min_pairs": CFG["RUN_A_FILTER_MIN_PAIRS"]},
    {"tag": CFG["RUN_B_TAG"], "gpu": 1, "start_ratio": CFG["RUN_B_FILTER_START_RATIO"], "end_ratio": CFG["RUN_B_FILTER_END_RATIO"], "start_epoch": CFG["RUN_B_FILTER_START_EPOCH"], "end_epoch": CFG["RUN_B_FILTER_END_EPOCH"], "min_pairs": CFG["RUN_B_FILTER_MIN_PAIRS"]},
]

jobs = []
for cfg in configs:
    save_path = f"{cfg['tag']}_{RUN_SUFFIX}_{commit}"
    log_path = run_logs_dir / f"{save_path}.log"
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(cfg["gpu"])
    env.update({
        "DATA_ROOT": CFG["DATA_ROOT"], "REGDB_SOURCE": CFG["REGDB_SOURCE"],
        "RUN_NAME": save_path, "DEVICE": "0", "SEED": str(CFG["SEED"]), "TRIAL": str(CFG["TRIAL"]),
        "STAGE2_EPOCH": str(CFG["STAGE2_EPOCH"]), "MILESTONES": CFG["MILESTONES"],
        "BATCH_PIDNUM": str(CFG["BATCH_PIDNUM"]), "PID_NUMSAMPLE": str(CFG["PID_NUMSAMPLE"]),
        "TEST_BATCH": str(CFG["TEST_BATCH"]), "NUM_WORKERS": str(CFG["NUM_WORKERS"]), "LR": str(CFG["LR"]),
        "PHASE1_CKPT": str(phase1_ckpt),
        "UPR_BETA": str(CFG["UPR_BETA"]), "UPR_GAMMA": str(CFG["UPR_GAMMA"]),
        "UPR_MARGIN_WEIGHT": str(CFG["UPR_MARGIN_WEIGHT"]), "UPR_WARMUP_EPOCH": str(CFG["UPR_WARMUP_EPOCH"]),
        "UPR_FILTER_START_RATIO": str(cfg["start_ratio"]), "UPR_FILTER_END_RATIO": str(cfg["end_ratio"]),
        "UPR_FILTER_START_EPOCH": str(cfg["start_epoch"]), "UPR_FILTER_END_EPOCH": str(cfg["end_epoch"]),
        "UPR_FILTER_MIN_PAIRS": str(cfg["min_pairs"]),
    })

    print("Starting", cfg["tag"], "on GPU", cfg["gpu"])
    print("save_path:", save_path)
    print("log_path:", log_path)
    f = open(log_path, "w")
    p = subprocess.Popen(["bash", "scripts/run_regdb_upr_v02_filter.sh"], cwd=wsl_dir, env=env, stdout=f, stderr=subprocess.STDOUT)
    jobs.append({"tag": cfg["tag"], "pid": p.pid, "process": p, "save_path": save_path, "log_path": str(log_path), "gpu": cfg["gpu"], "filter_start_ratio": cfg["start_ratio"], "filter_end_ratio": cfg["end_ratio"], "filter_start_epoch": cfg["start_epoch"], "filter_end_epoch": cfg["end_epoch"], "filter_min_pairs": cfg["min_pairs"]})

runtime = {"run_suffix": RUN_SUFFIX, "commit": commit, "phase1_ckpt": str(phase1_ckpt), "jobs": [{k:v for k,v in j.items() if k != "process"} for j in jobs], "references": {"baseline_p2s15": {"r1": CFG["REF_BASELINE_P2S15_R1"], "map": CFG["REF_BASELINE_P2S15_MAP"], "minp": CFG["REF_BASELINE_P2S15_MINP"]}, "upr_v01_p2s15": {"r1": CFG["REF_UPR_V01_P2S15_R1"], "map": CFG["REF_UPR_V01_P2S15_MAP"], "minp": CFG["REF_UPR_V01_P2S15_MINP"]}}}
Path("/kaggle/working/step7_runtime_info.json").write_text(json.dumps(runtime, indent=2))
print("Started jobs:")
for j in jobs:
    print(j["tag"], "PID", j["pid"], "log", j["log_path"])


Starting upr_v02_b01_g00_filter075to100_p2s15 on GPU 0
save_path: upr_v02_b01_g00_filter075to100_p2s15_step7_20260623_135458_f63d8f5
log_path: /kaggle/working/run_logs/upr_v02_b01_g00_filter075to100_p2s15_step7_20260623_135458_f63d8f5.log
Starting upr_v02_b01_g00_filter055to100_p2s15 on GPU 1
save_path: upr_v02_b01_g00_filter055to100_p2s15_step7_20260623_135458_f63d8f5
log_path: /kaggle/working/run_logs/upr_v02_b01_g00_filter055to100_p2s15_step7_20260623_135458_f63d8f5.log
Started jobs:
upr_v02_b01_g00_filter075to100_p2s15 PID 136 log /kaggle/working/run_logs/upr_v02_b01_g00_filter075to100_p2s15_step7_20260623_135458_f63d8f5.log
upr_v02_b01_g00_filter055to100_p2s15 PID 137 log /kaggle/working/run_logs/upr_v02_b01_g00_filter055to100_p2s15_step7_20260623_135458_f63d8f5.log


## Block 6 - Monitor jobs. Re-run this cell while they are running.

In [7]:
subprocess.run(["nvidia-smi"])
info = json.loads(Path("/kaggle/working/step7_runtime_info.json").read_text())
for job in info["jobs"]:
    print("\n==============================")
    print(job["tag"])
    print(job["log_path"])
    print("==============================")
    p = Path(job["log_path"])
    if p.exists():
        subprocess.run(["tail", "-40", str(p)])
    else:
        print("log not found yet")


Tue Jun 23 13:55:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Block 7 — Wait for jobs to finish.

In [15]:
if "jobs" in globals():
    for job in jobs:
        ret = job["process"].wait()
        print(job["tag"], "finished with code", ret)
else:
    print("No jobs variable found. Check process list manually.")
    subprocess.run("ps -ef | grep 'python main.py' | grep -v grep", shell=True)


upr_v02_b01_g00_filter075to100_p2s15 finished with code 0
upr_v02_b01_g00_filter055to100_p2s15 finished with code 0


## Block 8 - Collect metrics, relation stats, and comparison CSV

This polls the two background PIDs from Block 6. You can stop this cell if you only want manual monitoring.

In [16]:
# 8. Collect metrics and relation summaries.
info = json.loads(Path("/kaggle/working/step7_runtime_info.json").read_text())

def parse_best_metrics(log_text):
    pattern = re.compile(r"R1:([0-9.]+);\s+R10:([0-9.]+);\s+R20:([0-9.]+);\s+mAP:([0-9.]+);\s+mINP:([0-9.]+)")
    matches = pattern.findall(log_text)
    if not matches:
        return "", "", "", "", ""
    return max(matches, key=lambda x: float(x[3]))

rows = [
    {"run": "reference_baseline_p2s15_step6", "stage2_epoch": 15, "filter_start_ratio": "", "filter_end_ratio": "", "best_r1_by_map": CFG["REF_BASELINE_P2S15_R1"], "best_map": CFG["REF_BASELINE_P2S15_MAP"], "best_minp": CFG["REF_BASELINE_P2S15_MINP"], "last_common_accuracy": "", "last_mutual_match_ratio": "", "last_num_common": "", "last_num_specific": "", "last_num_remain": ""},
    {"run": "reference_upr_v01_b01_g00_w2_p2s15_step6", "stage2_epoch": 15, "filter_start_ratio": "", "filter_end_ratio": "", "best_r1_by_map": CFG["REF_UPR_V01_P2S15_R1"], "best_map": CFG["REF_UPR_V01_P2S15_MAP"], "best_minp": CFG["REF_UPR_V01_P2S15_MINP"], "last_common_accuracy": "", "last_mutual_match_ratio": "", "last_num_common": "", "last_num_specific": "", "last_num_remain": ""},
]

for job in info["jobs"]:
    run_dir = wsl_dir.parent / "saved_regdb_resnet" / f"{job['save_path']}_1"
    log_path = run_dir / "log" / "log.txt"
    stats_dir = run_dir / "relation_stats"
    stats_csv = stats_dir / "relation_stats_summary.csv"
    if stats_dir.exists():
        subprocess.run(["python", "scripts/collect_relation_stats.py", "--stats-dir", str(stats_dir), "--csv-output", str(stats_csv)], cwd=wsl_dir, check=True)
    text = log_path.read_text() if log_path.exists() else ""
    r1, r10, r20, map_, minp = parse_best_metrics(text)
    last_stats = {}
    if stats_csv.exists():
        with stats_csv.open() as f:
            data = list(csv.DictReader(f))
            if data: last_stats = data[-1]
    rows.append({"run": job["tag"], "stage2_epoch": CFG["STAGE2_EPOCH"], "filter_start_ratio": job["filter_start_ratio"], "filter_end_ratio": job["filter_end_ratio"], "best_r1_by_map": r1, "best_map": map_, "best_minp": minp, "last_common_accuracy": last_stats.get("common_accuracy", ""), "last_mutual_match_ratio": last_stats.get("mutual_match_ratio", ""), "last_num_common": last_stats.get("num_common", ""), "last_num_specific": last_stats.get("num_specific", ""), "last_num_remain": last_stats.get("num_remain", "")})

out = Path("/kaggle/working/step7_v02_filter_curriculum_summary.csv")
with out.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader(); writer.writerows(rows)
print(out.read_text())
print("Saved:", out)


Wrote /kaggle/working/UIT.CS2309/saved_regdb_resnet/upr_v02_b01_g00_filter075to100_p2s15_step7_20260623_135458_f63d8f5_1/relation_stats/relation_stats_summary.csv
{'epoch': 0, 'num_r2i_pairs': 142, 'num_i2r_pairs': 124, 'num_common': 47, 'num_specific': 32, 'num_remain': 99, 'mutual_match_ratio': 0.33098591549295775, 'r2i_pair_accuracy': 0.3732394366197183, 'i2r_pair_accuracy': 0.3467741935483871, 'common_accuracy': 0.5957446808510638, 'rgb_entropy_mean': 0.17003308491088964, 'ir_entropy_mean': 0.18085551654863424, 'rgb_margin_mean': 0.5666432981771526, 'ir_margin_mean': 0.5349934934331859, 'proto_common_mean': 0.4523059738443253, 'proto_specific_mean': 0.2990252198651433, 'proto_remain_mean': 0.3325949123110434}
{'epoch': 1, 'num_r2i_pairs': 151, 'num_i2r_pairs': 132, 'num_common': 59, 'num_specific': 32, 'num_remain': 95, 'mutual_match_ratio': 0.39072847682119205, 'r2i_pair_accuracy': 0.46357615894039733, 'i2r_pair_accuracy': 0.4090909090909091, 'common_accuracy': 0.6610169491525424,

## Block 9 - Archive small artifacts.

In [17]:
artifact_dir = Path("/kaggle/working/artifacts_step7_v02_filter")
if artifact_dir.exists(): shutil.rmtree(artifact_dir)
(artifact_dir / "logs").mkdir(parents=True, exist_ok=True)
(artifact_dir / "relation_stats").mkdir(parents=True, exist_ok=True)
for p in [Path("/kaggle/working/step7_runtime_info.json"), Path("/kaggle/working/step7_v02_filter_curriculum_summary.csv")]:
    if p.exists(): shutil.copy2(p, artifact_dir / p.name)
info = json.loads(Path("/kaggle/working/step7_runtime_info.json").read_text())
for job in info["jobs"]:
    run_dir = wsl_dir.parent / "saved_regdb_resnet" / f"{job['save_path']}_1"
    log_file = run_dir / "log" / "log.txt"
    if log_file.exists(): shutil.copy2(log_file, artifact_dir / "logs" / f"{job['save_path']}_log.txt")
    stats_csv = run_dir / "relation_stats" / "relation_stats_summary.csv"
    if stats_csv.exists(): shutil.copy2(stats_csv, artifact_dir / "relation_stats" / f"{job['save_path']}_relation_stats_summary.csv")
if Path("/kaggle/working/run_logs").exists(): shutil.copytree(Path("/kaggle/working/run_logs"), artifact_dir / "run_logs", dirs_exist_ok=True)
tar_path = Path("/kaggle/working/artifacts_step7_v02_filter.tar.gz")
if tar_path.exists(): tar_path.unlink()
subprocess.run(["tar", "-czf", str(tar_path), "-C", "/kaggle/working", artifact_dir.name], check=True)
print("Artifact:", tar_path)
print("Size MB:", tar_path.stat().st_size / (1024*1024))


Artifact: /kaggle/working/artifacts_step7_v02_filter.tar.gz
Size MB: 0.014238357543945312


## Block 10 - Optional: upload final artifact to GitHub Release
This is disabled by default. Use it only if you want persistence outside Kaggle.
It uploads /kaggle/working/artifacts_step5b_t4x2.tar.gz as a release asset.

In [18]:
from pathlib import Path
from kaggle_secrets import UserSecretsClient
import os
import json
import shutil
import subprocess
import textwrap
import hashlib
import urllib.request
import urllib.error
from datetime import datetime

# =========================
# Backup config
# =========================
PUSH_SMALL_BACKUP_TO_GIT = True
UPLOAD_TAR_TO_RELEASE = False  # keep False unless your GITHUB_TOKEN works with GitHub Releases API

OWNER = CFG.get("GITHUB_OWNER", "TranTruongMMCII")
REPO_NAME = CFG.get("REPO_NAME", "UIT.CS2309")
BRANCH = CFG.get("BRANCH", "feature/upr-cre")
WORK_DIR = Path(CFG.get("WORK_DIR", "/kaggle/working"))
REPO_DIR = WORK_DIR / REPO_NAME

GITHUB_TOKEN_SECRET = CFG.get("GITHUB_TOKEN_SECRET", "GITHUB_TOKEN")
GIT_USER_NAME = CFG.get("GIT_USER_NAME", "Kaggle Bot")
GIT_USER_EMAIL = CFG.get("GIT_USER_EMAIL", "kaggle-bot@example.com")

ARTIFACT_DIR = WORK_DIR / "artifacts_step7_v02_filter"
ARTIFACT_TAR = WORK_DIR / "artifacts_step7_v02_filter.tar.gz"
RUNTIME_INFO = WORK_DIR / "step7_runtime_info.json"
SUMMARY_CSV = WORK_DIR / "step7_v02_filter_curriculum_summary.csv"

INCLUDE_LARGE_FILES_IN_GIT = False
MAX_NORMAL_GIT_FILE_MB = 90

# =========================
# Helpers
# =========================
def run(cmd, cwd=REPO_DIR, env=None, check=True):
    print("+", " ".join(map(str, cmd)))
    return subprocess.run(cmd, cwd=cwd, env=env, check=check)

def sha256_file(path: Path, chunk_size: int = 1024 * 1024):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def copy_if_exists(src: Path, dst: Path):
    if not src.exists():
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)
    return True

# =========================
# Validate Step 7 outputs
# =========================
if not ARTIFACT_DIR.exists() and not ARTIFACT_TAR.exists():
    raise FileNotFoundError(
        "Missing Step 7 artifact. Expected one of:\n"
        f"- {ARTIFACT_DIR}\n"
        f"- {ARTIFACT_TAR}\n"
        "Run Block 9 archive first."
    )

if not RUNTIME_INFO.exists():
    raise FileNotFoundError(f"Missing runtime info: {RUNTIME_INFO}")

runtime_info = json.loads(RUNTIME_INFO.read_text())
run_id = runtime_info.get("run_suffix", "")
if not run_id:
    commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR).decode().strip()
    run_id = f"step7_manual_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{commit}"
    runtime_info["run_suffix"] = run_id

print("Run ID:", run_id)
print("Artifact dir:", ARTIFACT_DIR if ARTIFACT_DIR.exists() else None)
print("Artifact tar:", ARTIFACT_TAR if ARTIFACT_TAR.exists() else None)

# =========================
# 10A. Push small artifacts using old git push method
# =========================
if PUSH_SMALL_BACKUP_TO_GIT:
    token = UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET).strip()

    askpass = WORK_DIR / "git_askpass_push_step7.sh"
    askpass.write_text(textwrap.dedent(f"""\
#!/bin/sh
case "$1" in
  *Username*) echo "{OWNER}" ;;
  *Password*) echo "{token}" ;;
  *) echo "" ;;
esac
"""))
    askpass.chmod(0o700)

    env = os.environ.copy()
    env["GIT_ASKPASS"] = str(askpass)
    env["GIT_TERMINAL_PROMPT"] = "0"

    public_url = f"https://github.com/{OWNER}/{REPO_NAME}.git"

    try:
        run(["git", "config", "user.name", GIT_USER_NAME], env=env)
        run(["git", "config", "user.email", GIT_USER_EMAIL], env=env)
        run(["git", "remote", "set-url", "origin", public_url], env=env)
        run(["git", "fetch", "origin", BRANCH], env=env)
        run(["git", "checkout", BRANCH], env=env)
        run(["git", "pull", "--rebase", "origin", BRANCH], env=env)

        backup_dir = REPO_DIR / "experiments" / "kaggle_runs" / run_id
        if backup_dir.exists():
            shutil.rmtree(backup_dir)
        backup_dir.mkdir(parents=True, exist_ok=True)

        # Runtime and summary
        (backup_dir / "runtime_info.json").write_text(json.dumps(runtime_info, indent=2, ensure_ascii=False))
        copy_if_exists(SUMMARY_CSV, backup_dir / SUMMARY_CSV.name)

        # Copy small artifacts from archive folder
        if ARTIFACT_DIR.exists():
            for subdir in ["logs", "run_logs", "relation_stats"]:
                src = ARTIFACT_DIR / subdir
                if src.exists():
                    copy_if_exists(src, backup_dir / subdir)
            for pattern in ["*.csv", "*.txt", "*.json"]:
                for src in ARTIFACT_DIR.glob(pattern):
                    copy_if_exists(src, backup_dir / src.name)

        # Copy global run logs if present
        global_run_logs = WORK_DIR / "run_logs"
        if global_run_logs.exists():
            copy_if_exists(global_run_logs, backup_dir / "run_logs")

        # Manifest for large files
        large_files = []
        if ARTIFACT_TAR.exists():
            large_files.append(ARTIFACT_TAR)
        if ARTIFACT_DIR.exists():
            for p in ARTIFACT_DIR.rglob("*"):
                if p.is_file() and p.suffix in [".pth", ".pt", ".ckpt", ".gz"]:
                    large_files.append(p)

        manifest = {
            "run_id": run_id,
            "repo": f"{OWNER}/{REPO_NAME}",
            "branch": BRANCH,
            "backup_dir_in_repo": str(backup_dir.relative_to(REPO_DIR)),
            "artifact_dir": str(ARTIFACT_DIR) if ARTIFACT_DIR.exists() else None,
            "artifact_tar": str(ARTIFACT_TAR) if ARTIFACT_TAR.exists() else None,
            "large_files": [],
            "note": "Large files are not committed by default. Upload .pth/.tar.gz to Kaggle Dataset/Input if needed across sessions.",
        }

        large_dst_dir = backup_dir / "large_files"
        large_dst_dir.mkdir(parents=True, exist_ok=True)

        for src in sorted(set(large_files)):
            size_mb = src.stat().st_size / (1024 * 1024)
            item = {
                "source_path": str(src),
                "name": src.name,
                "size_mb": round(size_mb, 2),
                "sha256": sha256_file(src) if size_mb <= 1024 else "skipped_for_large_file",
                "committed_to_git": False,
            }
            if INCLUDE_LARGE_FILES_IN_GIT and size_mb <= MAX_NORMAL_GIT_FILE_MB:
                dst = large_dst_dir / src.name
                shutil.copy2(src, dst)
                item["committed_to_git"] = True
                item["repo_path"] = str(dst.relative_to(REPO_DIR))
            elif INCLUDE_LARGE_FILES_IN_GIT and size_mb > MAX_NORMAL_GIT_FILE_MB:
                item["skip_reason"] = f"larger than {MAX_NORMAL_GIT_FILE_MB} MB"
            manifest["large_files"].append(item)

        (backup_dir / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
        (backup_dir / "README.md").write_text(f"""# Kaggle run backup: {run_id}

This directory was pushed from Kaggle using normal `git push`.

## Contents

- Small logs, CSV summaries, relation statistics, and manifest files.
- Large files such as `.pth` or `.tar.gz` are recorded in `manifest.json`.

## Large artifact policy

By default, checkpoints and full `.tar.gz` artifacts are **not committed** into git.
Upload those files to Kaggle Dataset/Input if they are needed across independent Kaggle sessions.
""")

        print("Prepared backup directory:")
        for p in sorted(backup_dir.rglob("*")):
            if p.is_file():
                print(" -", p.relative_to(REPO_DIR), f"({p.stat().st_size/1024:.1f} KiB)")

        rel_backup = backup_dir.relative_to(REPO_DIR)
        run(["git", "add", "-f", str(rel_backup)], env=env)

        diff_status = subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=REPO_DIR, env=env)
        if diff_status.returncode == 0:
            print("No staged changes to commit.")
        else:
            run(["git", "commit", "-m", f"exp: add Step 7 Kaggle run backup {run_id}"], env=env)
            run(["git", "push", "origin", BRANCH], env=env)

        print("Small backup pushed to:")
        print(f"https://github.com/{OWNER}/{REPO_NAME}/tree/{BRANCH}/experiments/kaggle_runs/{run_id}")
    finally:
        try:
            askpass.unlink()
        except FileNotFoundError:
            pass
else:
    print("Small git backup disabled.")

# =========================
# 10B. Optional: upload full tar.gz to GitHub Release
# =========================
if UPLOAD_TAR_TO_RELEASE:
    if not ARTIFACT_TAR.exists():
        raise FileNotFoundError(f"Missing artifact tar for release upload: {ARTIFACT_TAR}")

    token = UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET).strip()
    tag = f"kaggle-{run_id}"
    artifact = ARTIFACT_TAR

    headers_json = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json",
        "Content-Type": "application/json",
        "X-GitHub-Api-Version": "2022-11-28",
        "User-Agent": "kaggle-upr-cre-backup",
    }

    def api_json(url, method="GET", data=None, headers=None):
        req = urllib.request.Request(url, data=data, method=method, headers=headers or headers_json)
        with urllib.request.urlopen(req) as resp:
            body = resp.read().decode()
            return json.loads(body) if body else {}

    # Get existing release, or create one.
    release = None
    get_url = f"https://api.github.com/repos/{OWNER}/{REPO_NAME}/releases/tags/{tag}"
    try:
        release = api_json(get_url)
        print("Existing release found:", release["html_url"])
    except urllib.error.HTTPError as e:
        if e.code != 404:
            print(e.read().decode())
            raise

    if release is None:
        create_url = f"https://api.github.com/repos/{OWNER}/{REPO_NAME}/releases"
        payload = json.dumps({
            "tag_name": tag,
            "name": f"Kaggle Step 7 {tag}",
            "body": "Step 7 UPR-CRE v0.2 artifacts generated from Kaggle.",
            "draft": False,
            "prerelease": False,
        }).encode()
        release = api_json(create_url, method="POST", data=payload)
        print("Created release:", release["html_url"])

    upload_url = release["upload_url"].split("{")[0] + f"?name={artifact.name}"
    headers_upload = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json",
        "Content-Type": "application/gzip",
        "X-GitHub-Api-Version": "2022-11-28",
        "User-Agent": "kaggle-upr-cre-backup",
    }
    req = urllib.request.Request(upload_url, data=artifact.read_bytes(), method="POST", headers=headers_upload)
    with urllib.request.urlopen(req) as resp:
        asset = json.loads(resp.read().decode())
    print("Release:", release["html_url"])
    print("Asset:", asset["browser_download_url"])
else:
    print("GitHub Release upload disabled. Set UPLOAD_TAR_TO_RELEASE=True in this block to use it.")


Run ID: step7_20260623_135458
Artifact dir: /kaggle/working/artifacts_step7_v02_filter
Artifact tar: /kaggle/working/artifacts_step7_v02_filter.tar.gz
+ git config user.name Kaggle Bot
+ git config user.email kaggle-bot@example.com
+ git remote set-url origin https://github.com/TranTruongMMCII/UIT.CS2309.git
+ git fetch origin feature/upr-cre


From https://github.com/TranTruongMMCII/UIT.CS2309
 * branch            feature/upr-cre -> FETCH_HEAD
Already on 'feature/upr-cre'


+ git checkout feature/upr-cre
Your branch is up to date with 'origin/feature/upr-cre'.
+ git pull --rebase origin feature/upr-cre


From https://github.com/TranTruongMMCII/UIT.CS2309
 * branch            feature/upr-cre -> FETCH_HEAD


Already up to date.
Prepared backup directory:
 - experiments/kaggle_runs/step7_20260623_135458/README.md (0.5 KiB)
 - experiments/kaggle_runs/step7_20260623_135458/logs/upr_v02_b01_g00_filter055to100_p2s15_step7_20260623_135458_f63d8f5_log.txt (11.7 KiB)
 - experiments/kaggle_runs/step7_20260623_135458/logs/upr_v02_b01_g00_filter075to100_p2s15_step7_20260623_135458_f63d8f5_log.txt (11.7 KiB)
 - experiments/kaggle_runs/step7_20260623_135458/manifest.json (0.7 KiB)
 - experiments/kaggle_runs/step7_20260623_135458/relation_stats/upr_v02_b01_g00_filter055to100_p2s15_step7_20260623_135458_f63d8f5_relation_stats_summary.csv (3.6 KiB)
 - experiments/kaggle_runs/step7_20260623_135458/relation_stats/upr_v02_b01_g00_filter075to100_p2s15_step7_20260623_135458_f63d8f5_relation_stats_summary.csv (3.6 KiB)
 - experiments/kaggle_runs/step7_20260623_135458/run_logs/upr_v02_b01_g00_filter055to100_p2s15_step7_20260623_135458_f63d8f5.log (22.9 KiB)
 - experiments/kaggle_runs/step7_20260623_135458/run_lo

To https://github.com/TranTruongMMCII/UIT.CS2309.git
   aff642d..22127c7  feature/upr-cre -> feature/upr-cre
